# Experiment: Internal Throughput Sweep

Objective:
- Compare `eta_internal(lambda)` for 3/7/19-port lanterns.
- Sweep `alpha_ad` and `eta0` for one representative design.
- Inspect how `sigma_mix` changes port power redistribution.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config_loader import load_config_bundle
from src.lantern_surrogate import LanternInternalModel
from src.sky_background import sky_photon_flux_per_nm_arcsec2
from src.snr import stacked_snr
from src.system_model import SystemModel

config = load_config_bundle(REPO_ROOT / 'config')
lam_nm = config['lam_nm']
base = config['base']
plt.rcParams['figure.dpi'] = 140


In [ ]:
port_list = [3, 7, 19]
fig, ax = plt.subplots(figsize=(8, 4.5))
for n_port in port_list:
    model = LanternInternalModel(
        n_port=n_port,
        lambda0_nm=base['lantern']['lambda0_nm'],
        alpha_ad=base['lantern']['alpha_ad'],
        eta0=base['lantern']['eta0'],
        sigma_mix=base['lantern']['sigma_mix'],
        w_cut=base['lantern']['w_cut'],
    )
    ax.plot(lam_nm, model.eta_internal(lam_nm), linewidth=2, label=f'{n_port} ports')

ax.set_xlabel('Wavelength [nm]')
ax.set_ylabel('Internal throughput')
ax.set_ylim(0.0, 1.05)
ax.set_title('eta_internal(lambda)')
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for alpha_ad in [0.1, 0.3, 0.6]:
    model = LanternInternalModel(
        n_port=7,
        lambda0_nm=base['lantern']['lambda0_nm'],
        alpha_ad=alpha_ad,
        eta0=base['lantern']['eta0'],
        sigma_mix=base['lantern']['sigma_mix'],
        w_cut=base['lantern']['w_cut'],
    )
    axes[0].plot(lam_nm, model.eta_internal(lam_nm), label=f'alpha_ad={alpha_ad}')

for eta0 in [0.85, 0.90, 0.95]:
    model = LanternInternalModel(
        n_port=7,
        lambda0_nm=base['lantern']['lambda0_nm'],
        alpha_ad=base['lantern']['alpha_ad'],
        eta0=eta0,
        sigma_mix=base['lantern']['sigma_mix'],
        w_cut=base['lantern']['w_cut'],
    )
    axes[1].plot(lam_nm, model.eta_internal(lam_nm), label=f'eta0={eta0}')

for ax in axes:
    ax.set_xlabel('Wavelength [nm]')
    ax.set_ylabel('Internal throughput')
    ax.set_ylim(0.0, 1.05)
    ax.grid(alpha=0.25)
    ax.legend()

axes[0].set_title('Sensitivity to alpha_ad')
axes[1].set_title('Sensitivity to eta0')
plt.tight_layout()


## Port redistribution example

`eta_internal` controls the surviving power fraction. `sigma_mix` changes how that surviving power is shuffled before the output port map is applied.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for ax, sigma_mix in zip(axes, [0.2, 0.8, 1.6]):
    model = LanternInternalModel(
        n_port=7,
        lambda0_nm=base['lantern']['lambda0_nm'],
        alpha_ad=base['lantern']['alpha_ad'],
        eta0=base['lantern']['eta0'],
        sigma_mix=sigma_mix,
        w_cut=base['lantern']['w_cut'],
        port_map_mode='random_fixed',
        random_seed=42,
    )
    payload = model.propagate_power(model.generate_input_modes(21), np.array([950.0]))
    ax.bar(np.arange(1, model.n_port + 1), payload['p_out'])
    ax.set_title(f'sigma_mix={sigma_mix}')
    ax.set_xlabel('Port index')
    ax.grid(alpha=0.25, axis='y')
axes[0].set_ylabel('Output power fraction')
plt.tight_layout()
